# 🕐 Notebook 2 — Ambulance ETA Predictor

**Project:** AI-Enabled Smart Emergency Response & Ambulance Coordination System
**Model role:** Given a candidate ambulance + route conditions, predict the seconds-to-arrival.
This score lets the dispatch engine pick the *fastest-effective* ambulance, not just the closest.

### 🎯 Performance Target
- **R² ≥ 0.95**
- **MAE ≤ 30 s** (a half-minute is the threshold below which dispatchers stop double-checking)
- **MAPE ≤ 8 %**

### Why this is a regression problem (not classification)
ETA is a continuous number with heavy-tailed distribution (rural vs city calls). We use
gradient-boosted trees on engineered features and an ensemble for variance reduction.

## 1 · Setup & Imports

In [ ]:
# !pip install -q numpy pandas scikit-learn xgboost lightgbm catboost shap matplotlib seaborn joblib

import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("Setup complete ✔")

## 2 · Synthetic Data Generation

The ground-truth ETA is built from a physics-style equation:

`eta = (distance / effective_speed) × congestion_factor × weather_factor × ambulance_factor × road_factor + lognormal_noise`

This gives boosters strong non-linear interactions to learn (eg. distance × congestion).

In [ ]:
N = 50_000

def generate_eta_dataset(n=N):
    rng = np.random.default_rng(RANDOM_STATE)

    distance_km   = np.round(rng.gamma(shape=2.2, scale=3.4, size=n).clip(0.3, 60), 2)
    hour          = rng.integers(0, 24, n)
    dow           = rng.integers(0, 7,  n)
    is_weekend    = (dow >= 5).astype(int)
    is_rush       = ((dow < 5) & (((hour >= 8) & (hour <= 10)) |
                                  ((hour >= 17) & (hour <= 20)))).astype(int)
    is_night      = ((hour >= 23) | (hour <= 5)).astype(int)

    # Congestion 0–1, biased by rush hour
    congestion    = np.clip(
        rng.beta(2 + 5*is_rush, 6) * (1 - 0.5*is_night),
        0, 1).round(3)

    # Weather: 0=clear, 1=rain, 2=heavy rain, 3=storm
    weather       = rng.choice([0,1,2,3], size=n, p=[0.62, 0.22, 0.12, 0.04])
    weather_mult  = np.array([1.0, 1.10, 1.25, 1.55])[weather]

    # Ambulance type: 0=BLS, 1=ALS, 2=ICU mobile (heavier vehicle = slightly slower)
    amb_type      = rng.choice([0,1,2], size=n, p=[0.55, 0.35, 0.10])
    amb_mult      = np.array([1.00, 1.02, 1.06])[amb_type]

    # Road type: 0=urban, 1=highway, 2=rural
    road_type     = rng.choice([0,1,2], size=n, p=[0.55, 0.25, 0.20])
    base_speed_kmh = np.array([35, 80, 55])[road_type]   # free-flow speed

    # Effective speed after congestion (highways degrade more under congestion)
    cong_penalty   = np.array([0.55, 0.75, 0.40])[road_type]
    eff_speed_kmh  = base_speed_kmh * (1 - congestion * cong_penalty)
    eff_speed_kmh  = np.clip(eff_speed_kmh, 8, base_speed_kmh)

    # Pure travel time in seconds
    travel_sec     = (distance_km / eff_speed_kmh) * 3600

    # Apply multiplicative effects
    eta = travel_sec * weather_mult * amb_mult

    # Add realistic dispatch overhead 30–90 s
    eta += rng.uniform(30, 90, n)
    # Multiplicative log-normal noise (~6–8 %)
    eta *= rng.lognormal(mean=0, sigma=0.06, size=n)
    eta = np.round(eta, 1)

    return pd.DataFrame({
        "distance_km": distance_km,
        "congestion": congestion,
        "hour": hour,
        "day_of_week": dow,
        "is_weekend": is_weekend,
        "is_rush_hour": is_rush,
        "is_night": is_night,
        "weather": weather,
        "ambulance_type": amb_type,
        "road_type": road_type,
        "eta_seconds": eta,
    })

df = generate_eta_dataset()
print(f"Generated {len(df):,} rows × {df.shape[1]} columns")
df.head()

## 3 · Exploratory Data Analysis

### 3.1 · Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(df["eta_seconds"], bins=80, color="#2980b9", edgecolor="white")
axes[0].set_title("ETA distribution (seconds)")
axes[0].set_xlabel("ETA (s)"); axes[0].set_ylabel("count")
axes[0].axvline(df["eta_seconds"].median(), color="red", ls="--",
                label=f"median = {df['eta_seconds'].median():.0f}s")
axes[0].legend()

axes[1].hist(np.log1p(df["eta_seconds"]), bins=80, color="#16a085", edgecolor="white")
axes[1].set_title("log(ETA) — much closer to normal")
axes[1].set_xlabel("log(ETA+1)")
plt.tight_layout(); plt.show()

print(df["eta_seconds"].describe().round(1))

### 3.2 · Distance vs ETA

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sample = df.sample(5000, random_state=1)
sc = ax.scatter(sample["distance_km"], sample["eta_seconds"],
                c=sample["congestion"], cmap="RdYlGn_r", s=10, alpha=0.6)
plt.colorbar(sc, ax=ax, label="congestion")
ax.set_xlabel("distance (km)"); ax.set_ylabel("ETA (s)")
ax.set_title("Distance vs ETA — colour = congestion")
plt.tight_layout(); plt.show()

### 3.3 · Hourly ETA pattern

In [ ]:
hourly = df.groupby("hour")["eta_seconds"].agg(["mean","median"])
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(hourly.index, hourly["mean"],   label="mean",   color="#c0392b", marker="o")
ax.plot(hourly.index, hourly["median"], label="median", color="#2c3e50", marker="o")
ax.fill_between([8,10],   hourly.values.min(), hourly.values.max(),
                alpha=0.12, color="orange", label="rush hour")
ax.fill_between([17,20], hourly.values.min(), hourly.values.max(),
                alpha=0.12, color="orange")
ax.set_xticks(range(0,24)); ax.set_xlabel("hour of day"); ax.set_ylabel("ETA (s)")
ax.set_title("ETA inflates during morning + evening rush")
ax.legend(); plt.tight_layout(); plt.show()

### 3.4 · Categorical breakdowns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.boxplot(x="weather",   y="eta_seconds", data=df, ax=axes[0],
            palette="Blues")
axes[0].set_title("ETA by weather (0=clear … 3=storm)")
sns.boxplot(x="ambulance_type", y="eta_seconds", data=df, ax=axes[1],
            palette="Purples")
axes[1].set_title("ETA by ambulance type (0=BLS,1=ALS,2=ICU)")
sns.boxplot(x="road_type", y="eta_seconds", data=df, ax=axes[2],
            palette="Greens")
axes[2].set_title("ETA by road type (0=urban,1=highway,2=rural)")
plt.tight_layout(); plt.show()

### 3.5 · Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson correlation")
plt.tight_layout(); plt.show()

## 4 · Feature Engineering

Tree models love interaction features made explicit.

In [ ]:
def add_features(d):
    d = d.copy()
    # Free-flow speed proxy by road type
    base_speed = np.array([35, 80, 55])[d["road_type"].values]
    d["base_speed_kmh"]  = base_speed
    d["est_free_flow_s"] = (d["distance_km"] / base_speed) * 3600
    d["distance_x_congestion"] = d["distance_km"] * d["congestion"]
    d["log_distance"]         = np.log1p(d["distance_km"])
    return d

df_fe = add_features(df)
feature_cols = [c for c in df_fe.columns if c != "eta_seconds"]
print("Final feature count:", len(feature_cols))
print(feature_cols)

## 5 · Train/Test Split & Scaling

`RobustScaler` is preferred here — heavy tails in ETA mean StandardScaler is misled by outliers.

In [ ]:
X = df_fe[feature_cols].values
y = df_fe["eta_seconds"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)

scaler   = RobustScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train_s.shape}  Test: {X_test_s.shape}")

## 6 · Model bake-off

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, do_cv=False):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    metrics = dict(
        MAE  = mean_absolute_error(y_te, preds),
        RMSE = np.sqrt(mean_squared_error(y_te, preds)),
        R2   = r2_score(y_te, preds),
        MAPE = mean_absolute_percentage_error(y_te, preds))
    if do_cv:
        cv_r2 = cross_val_score(model, X_tr, y_tr,
                                cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                                scoring="r2", n_jobs=-1).mean()
        metrics["CV_R2"] = cv_r2
    results[name] = {"model": model, "preds": preds, **metrics}
    print(f"{name:22s}  MAE={metrics['MAE']:7.2f}s  RMSE={metrics['RMSE']:7.2f}s  "
          f"R²={metrics['R2']:.4f}  MAPE={metrics['MAPE']*100:.2f}%"
          + (f"  CV-R²={metrics.get('CV_R2',0):.4f}" if do_cv else ""))
    return model

### 6.1 · Linear baseline

In [ ]:
evaluate("LinearRegression", LinearRegression(),
         X_train_s, y_train, X_test_s, y_test)
evaluate("Ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

### 6.2 · Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_depth=20,
                           min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)
evaluate("RandomForest", rf, X_train_s, y_train, X_test_s, y_test)

### 6.3 · XGBoost (tuned)

In [ ]:
xgb = XGBRegressor(
    n_estimators=900, max_depth=8, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=2, reg_alpha=0.1, reg_lambda=1.2,
    objective="reg:squarederror", tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1)
evaluate("XGBoost", xgb, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 6.4 · LightGBM (tuned)

In [ ]:
lgb = LGBMRegressor(
    n_estimators=900, num_leaves=63, max_depth=-1,
    learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=0.5,
    objective="regression", random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1)
evaluate("LightGBM", lgb, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 6.5 · CatBoost

In [ ]:
cat = CatBoostRegressor(
    iterations=900, depth=8, learning_rate=0.05,
    l2_leaf_reg=3.0, loss_function="RMSE",
    random_seed=RANDOM_STATE, verbose=False)
evaluate("CatBoost", cat, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 6.6 · Averaged ensemble of the 3 boosters

In [ ]:
preds_avg = (results["XGBoost"]["preds"] +
             results["LightGBM"]["preds"] +
             results["CatBoost"]["preds"]) / 3
metrics = dict(
    MAE  = mean_absolute_error(y_test, preds_avg),
    RMSE = np.sqrt(mean_squared_error(y_test, preds_avg)),
    R2   = r2_score(y_test, preds_avg),
    MAPE = mean_absolute_percentage_error(y_test, preds_avg))
results["EnsembleAvg(XGB+LGB+Cat)"] = {"model":"avg", "preds":preds_avg, **metrics}
print(f"EnsembleAvg                MAE={metrics['MAE']:7.2f}s  RMSE={metrics['RMSE']:7.2f}s  "
      f"R²={metrics['R2']:.4f}  MAPE={metrics['MAPE']*100:.2f}%")

## 7 · Leaderboard

In [ ]:
leaderboard = pd.DataFrame([
    {"model": k, "MAE": v["MAE"], "RMSE": v["RMSE"], "R2": v["R2"],
     "MAPE_%": v["MAPE"]*100}
    for k, v in results.items()
]).sort_values("R2", ascending=False).reset_index(drop=True)
display(leaderboard.round(4))

best_name = leaderboard.iloc[0]["model"]
print(f"\n🏆 Best: {best_name}  (R²={results[best_name]['R2']:.4f}, "
      f"MAE={results[best_name]['MAE']:.2f}s)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = leaderboard.sort_values("R2")
ax.barh(order["model"], order["R2"], color="#27ae60")
ax.axvline(0.95, ls="--", color="red", label="R²=0.95 target")
ax.set_xlim(0.5, 1.0); ax.set_xlabel("R²"); ax.legend()
ax.set_title("Model R² leaderboard")
plt.tight_layout(); plt.show()

## 8 · Diagnostics on the winning model

### 8.1 · Predicted vs Actual

In [ ]:
best_preds = results[best_name]["preds"]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, best_preds, s=4, alpha=0.35, color="#2980b9")
lims = [0, max(y_test.max(), best_preds.max())]
ax.plot(lims, lims, "r--", lw=2, label="perfect")
ax.set_xlabel("Actual ETA (s)"); ax.set_ylabel("Predicted ETA (s)")
ax.set_title(f"{best_name}: predicted vs actual"); ax.legend()
plt.tight_layout(); plt.show()

### 8.2 · Residuals

In [ ]:
residuals = y_test - best_preds
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(best_preds, residuals, s=4, alpha=0.35, color="#8e44ad")
axes[0].axhline(0, color="red", lw=1)
axes[0].set_xlabel("Predicted ETA (s)"); axes[0].set_ylabel("Residual (s)")
axes[0].set_title("Residuals vs predicted")

axes[1].hist(residuals, bins=80, color="#8e44ad", edgecolor="white")
axes[1].axvline(0, color="red"); axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual (s)")
plt.tight_layout(); plt.show()
print(f"Residual mean={residuals.mean():.2f}s  std={residuals.std():.2f}s")

### 8.3 · Error by distance bucket

In [ ]:
buckets = pd.cut(df_fe.iloc[len(X_train):]["distance_km"].values,
                 bins=[0,2,5,10,20,60],
                 labels=["0-2km","2-5km","5-10km","10-20km","20-60km"])
err_df = pd.DataFrame({"bucket": buckets, "abs_err": np.abs(residuals)})
agg = err_df.groupby("bucket")["abs_err"].agg(["mean","median","count"])
display(agg.round(2))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(x="bucket", y="abs_err", data=err_df, palette="Blues", ax=ax,
            showfliers=False)
ax.set_title("Absolute error by trip distance"); ax.set_ylabel("|error| (s)")
plt.tight_layout(); plt.show()

### 8.4 · Feature importance

In [ ]:
imp_model = results["XGBoost"]["model"]
imp = pd.Series(imp_model.feature_importances_, index=feature_cols)
fig, ax = plt.subplots(figsize=(9, 6))
imp.sort_values().plot.barh(ax=ax, color="#16a085")
ax.set_title("XGBoost feature importance")
plt.tight_layout(); plt.show()

### 8.5 · SHAP — global + local explanations

In [ ]:
import shap
try:
    explainer  = shap.TreeExplainer(imp_model)
    shap_vals  = explainer.shap_values(X_test_s[:1000])
    shap.summary_plot(shap_vals, X_test_s[:1000], feature_names=feature_cols, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## 9 · Try a fresh prediction

In [ ]:
def predict_eta(case: dict, model_name="XGBoost"):
    m = results[model_name]["model"]
    full = add_features(pd.DataFrame([case]))
    x = scaler.transform(full[feature_cols].values)
    return float(m.predict(x)[0])

scenarios = [
    {"distance_km":3.2,  "congestion":0.20, "hour":10, "day_of_week":2,
     "is_weekend":0, "is_rush_hour":1, "is_night":0,
     "weather":0, "ambulance_type":0, "road_type":0},
    {"distance_km":18.5, "congestion":0.65, "hour":18, "day_of_week":4,
     "is_weekend":0, "is_rush_hour":1, "is_night":0,
     "weather":2, "ambulance_type":1, "road_type":1},
    {"distance_km":1.4,  "congestion":0.05, "hour":3,  "day_of_week":6,
     "is_weekend":1, "is_rush_hour":0, "is_night":1,
     "weather":0, "ambulance_type":2, "road_type":0},
]

for s in scenarios:
    eta = predict_eta(s)
    print(f"distance={s['distance_km']}km cong={s['congestion']} → ETA = {eta:.0f}s "
          f"({eta/60:.1f} min)")

## 10 · Persist artefacts

In [ ]:
joblib.dump(results["XGBoost"]["model"],  "models/eta_xgb.pkl")
joblib.dump(results["LightGBM"]["model"], "models/eta_lgbm.pkl")
joblib.dump(results["CatBoost"]["model"], "models/eta_catboost.pkl")
joblib.dump(scaler,                       "models/eta_scaler.pkl")
joblib.dump(feature_cols,                 "models/eta_feature_cols.pkl")

final_metrics = {
    "best_model": best_name,
    "R2":   round(results[best_name]["R2"], 4),
    "MAE_seconds":  round(results[best_name]["MAE"], 2),
    "RMSE_seconds": round(results[best_name]["RMSE"], 2),
    "MAPE_pct":     round(results[best_name]["MAPE"]*100, 2),
}
with open("reports/eta_predictor_report.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print(json.dumps(final_metrics, indent=2))
print("\n✅ ETA predictor saved.")

## 11 · Summary

| Metric | Target | Achieved |
|--------|--------|----------|
| R²     | ≥ 0.95 | _see leaderboard_ |
| MAE    | ≤ 30 s | _see leaderboard_ |
| MAPE   | ≤ 8 %  | _see leaderboard_ |

**Why this works:**
- Engineered `est_free_flow_s` and `distance × congestion` give the boosters direct shortcuts.
- RobustScaler protects against the heavy ETA tail.
- Averaging 3 booster families wipes out individual variance.
- SHAP explains why a particular ETA was high (eg. heavy rain + rush hour) — useful for the dispatcher UI tooltip.

→ Continue to **Notebook 3 — Hospital Recommender**.